In [42]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from IPython.display import Image
from io import BytesIO
#from scrapy import fetch_website_links, fetch_website_contents
from openai import OpenAI

import requests
from bs4 import BeautifulSoup

import pandas as pd
import ast   # or json if it's valid JSON

from parsing_url import redacted_links, redacted_contents, \
    redacted_images, redacted_headings, redacted_metadata

In [43]:
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

API key looks good so far


#### Select Relative Links

In [ ]:
def get_url_system_prompt(name: str):
    system_prompt = f""" 
    You are provider of list of URLs related to {name}. 
    You must examine links and drop any that are not related to {name}. 
    Relations can include their work, family, controversies, good or bad work, 
    or criticism about them. Fetching content from url should not be greater then 500 words. 

    You should respond in JSON as in this example:
    {{
        "links": [
            {{"type": "about page", "url": "https://links", "summary": "Should not exceed 500 words"}}      
        ]
    }}
    """
    return system_prompt


def get_url_user_prompt(url: str, name: str):
    headers = {"User-Agent": "Mozilla/5.0"}
    links = redacted_links(url,headers)    
    user_prompt = f"""
    I am providing {url}. 
    Find a list of all links from {url} related to {name}.
    Analyze all links attached with {url}. 
    If a link contains any information about {name}, add it into the list; 
    if not, drop the link.
    Links found:
    {chr(10).join(links)}
    """
    return user_prompt


In [ ]:
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [46]:
def select_relevant_links(name: str, url: str):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": get_url_system_prompt(name=name)},
            {"role": "user", "content": get_url_user_prompt(url=url, name=name)}
        ],
        response_format={"type": "json_object"}
    )
    
    result = response.choices[0].message.content
    try:
        links = json.loads(result)
    except json.JSONDecodeError:
        links = {"links": []}  # fallback if parsing fails
    
    return links


In [47]:
relative_link = select_relevant_links(name='Albert Einstein',url='https://en.wikipedia.org/wiki/Albert_Einstein')

In [48]:
type(relative_link)

dict

In [49]:

df = pd.DataFrame(relative_link["links"])
print(df.head())

             type                                             url  \
0  biography page   https://en.wikipedia.org/wiki/Albert_Einstein   
1          family   https://en.wikipedia.org/wiki/Einstein_family   
2   family member      https://en.wikipedia.org/wiki/Mileva_Marić   
3   family member  https://en.wikipedia.org/wiki/Lieserl_Einstein   
4   family member    https://en.wikipedia.org/wiki/ Elsa_Einstein   

                                             summary  
0  Overview of Albert Einstein's life, major theo...  
1  Information about Einstein's family:\nMileva M...  
2  Mileva Marić, Albert Einstein's first wife and...  
3  Lieserl Einstein, Albert and Mileva's daughter...  
4  Elsa Löwenthal, Albert Einstein's wife and cou...  


#### fetches the page contents for each url, and appends the text into a new column <result>

In [50]:
def fetch_with_api(url: str, header: dict = None) -> str:
    """
    Fetch content from a given URL using the redacted_contents API.
    Returns the text content or an error message.
    """
    try:
        # Call your API function
        content = redacted_contents(url, header)
        return content if content else "No content returned"
    except Exception as e:
        return f"Error fetching {url}: {e}"

def enrich_dataframe_with_api(df: pd.DataFrame, header: dict = None) -> pd.DataFrame:
    """
    Append fetched content into a new <result> column using redacted_contents API.
    """
    df["result"] = df["url"].apply(lambda u: fetch_with_api(u, header))
    return df

In [51]:
# Suppose df has columns: type, url, summary
headers = {"User-Agent": "Mozilla/5.0"}   # optional, if your API requires headers

df = enrich_dataframe_with_api(df, header=headers)

# Inspect enriched data
print(df[["type", "url", "result"]].head())

             type                                             url  \
0  biography page   https://en.wikipedia.org/wiki/Albert_Einstein   
1          family   https://en.wikipedia.org/wiki/Einstein_family   
2   family member      https://en.wikipedia.org/wiki/Mileva_Marić   
3   family member  https://en.wikipedia.org/wiki/Lieserl_Einstein   
4   family member    https://en.wikipedia.org/wiki/ Elsa_Einstein   

                                              result  
0  [, Albert Einstein[a](14 March 1879–18 April 1...  
1  [, TheEinstein familyis the family of physicis...  
2  [, Mileva Marić(Serbian Cyrillic:Милева Марић,...  
3  [, TheEinstein familyis the family of physicis...  
4  [, Elsa Einstein(18 January 1876 – 20 December...  


In [52]:
row0_content = df.loc[0, "result"]
print(display(Markdown(row0_content[1])))

Albert Einstein[a](14 March 1879–18 April 1955) was a German-borntheoretical physicistbest known for developing thetheory of relativity. Einstein also made important contributions toquantum theory.[1][5]Hismass–energy equivalenceformulaE=mc2, which arises fromspecial relativity, has been called "the world's most famous equation".[6]He received the 1921Nobel Prize in Physicsfor "his services to theoretical physics, and especially for his discovery of the law of thephotoelectric effect".[7]

None


In [53]:
BIOGRAPHY_SYSTEM_PROMPT = '''
You are an expert biographer specializing in writing comprehensive, balanced, and insightful biographies of human beings. 
Your role is to analyze provided information and craft a narrative that highlights the essential characteristics of the person.

When writing a biography, you must cover the following aspects:

1. Identity & Background
   - Full name, birth date, birthplace
   - Family origins, early influences, and education

2. Personality Traits
   - Core qualities (discipline, creativity, resilience, empathy, etc.)
   - Strengths and weaknesses
   - How these traits shaped their decisions and achievements

3. Values & Beliefs
   - Moral compass and ethical stance
   - Religious, philosophical, or political views
   - Guiding principles in life and work

4. Professional Characteristics
   - Dedication and loyalty to their profession
   - Work ethic, leadership style, and innovations
   - Major contributions and achievements

5. Social & Family Characteristics
   - Relationships with family, friends, mentors, collaborators
   - Family sentiment toward them (supportive, conflicted, proud, unhappy)
   - Influence of family on their career and decisions

6. Controversies & Challenges
   - Criticisms, scandals, or wrong decisions
   - Political, social, or personal controversies
   - How they responded to adversity or mistakes

7. Impact & Legacy
   - Influence on society, culture, or future generations
   - Institutions, movements, or ideas they inspired
   - How they are remembered today

Your response must be factual, concise, and structured. 
If information is missing, clearly state what additional details are required.

You must output the biography in the following JSON format:

{
  "identity": {
    "full_name": "",
    "birth_date": "",
    "birthplace": "",
    "family_background": ""
  },
  "personality_traits": [
    "discipline", "creativity", "resilience"
  ],
  "values_beliefs": {
    "moral_compass": "",
    "religious_views": "",
    "philosophical_views": "",
    "political_views": ""
  },
  "professional_characteristics": {
    "dedication": "",
    "work_ethic": "",
    "leadership_style": "",
    "major_contributions": []
  },
  "social_family": {
    "relationships": "",
    "family_sentiment": "",
    "influence_on_career": ""
  },
  "controversies_challenges": [
    "criticism or scandal",
    "wrong decisions"
  ],
  "impact_legacy": {
    "societal_influence": "",
    "institutions_inspired": [],
    "how_remembered": ""
  }
}
'''


In [54]:
def build_user_prompts(df, name: str) -> list:
    """
    Build user prompts for biography writing using all rows in df['result'].
    :param df: Pandas DataFrame with 'type' and 'result' columns
    :param name: Person's name
    :return: List of user prompt strings
    """
    prompts = []
    for idx, row in df.iterrows():
        content = row["result"]
        prompt = f"""
Use the input parameter and write a biography on {name}.
Focus on identity, traits, values, professional life, family, controversies, and legacy.
Use the following content (type={row['type']}) as source material:
{content}
Return the biography in structured JSON format.
"""
        prompts.append(prompt)
    return prompts

In [ ]:
def create_biography(name,data_frame,model):
    prompts = build_user_prompts(data_frame, name)
    merged_prompt = "\n\n".join(prompts)
    response = openai.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": BIOGRAPHY_SYSTEM_PROMPT},
        {"role": "user", "content": merged_prompt}
    ],
    response_format={"type": "json_object"}
)
    
create_biography(name='A',data_frame = d,MODEL)

SyntaxError: positional argument follows keyword argument (3819443650.py, line 13)